# Week 2 – Day 11: Candidate Model Training & Comparison
**Project:** DubaiEstate AI 360 – Agentic AI Powered Real Estate Investment  
**Company:** NEOSKILLZ  
**Intern:** Nitesh Kumar Barnwal  
**Domain:** Artificial Intelligence & Machine Learning (AI/ML)  
**Branch:** `feature/week02-aiml-nitesh`  

---

## 1. Objective
Train an improved, non-linear regression model for Dubai real estate valuation and rigorously compare it against Day 10 baselines (`DummyRegressor` and `LinearRegression`):
1. Reuse the approved dataset (`131,283` eligible sales records), 13 model features, and the identical untouched test set (`26,257` transactions).
2. Select and train a high-performance candidate model: **RandomForestRegressor** with sensible regularized hyperparameters (`max_depth=12`, `min_samples_leaf=5`, `max_features='sqrt'`).
3. Evaluate out-of-sample performance using **MAE** and **RMSE**.
4. Compare against Day 10 baselines and demonstrate measurable error reduction.
5. Persist the full end-to-end preprocessing + model pipeline for Day 13 inference deployment.

In [1]:
import json
from pathlib import Path
import sys
import time

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline

# Local module resolution
CURRENT_DIR = Path.cwd()
if str(CURRENT_DIR) not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR))

from model_dataset import load_transactions
from preprocess import (
    ALL_FEATURES,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_preprocessor,
    check_transformed_matrix,
    fit_preprocessor,
    get_candidate_sales,
    prepare_features,
    transform_features,
)

print("All required packages and modules imported successfully.")

All required packages and modules imported successfully.


## 2. Load Dataset & Reproduce Day 10 Chronological Split
To ensure completely fair benchmarking, we reproduce the exact date-aware 80/20 train-test partition used in Day 10.

In [2]:
raw_df = load_transactions()
sales_df = get_candidate_sales(raw_df)

sales_df["INSTANCE_DATE"] = pd.to_datetime(sales_df["INSTANCE_DATE"], errors="coerce")
sales_df = sales_df.sort_values("INSTANCE_DATE").reset_index(drop=True)

split_idx = int(len(sales_df) * 0.80)
train_df = sales_df.iloc[:split_idx].copy()
test_df = sales_df.iloc[split_idx:].copy()

print(f"Total Eligible Sales: {len(sales_df):,} records")
print(f"Training Set:         {len(train_df):,} records ({train_df['INSTANCE_DATE'].min()} to {train_df['INSTANCE_DATE'].max()})")
print(f"Untouched Test Set:   {len(test_df):,} records ({test_df['INSTANCE_DATE'].min()} to {test_df['INSTANCE_DATE'].max()})")

# Verify alignment with Day 10 metrics.json
with open("metrics.json", "r", encoding="utf-8") as f:
    d10_metrics = json.load(f)

assert len(train_df) == d10_metrics["dataset"]["train_records"], "Train count mismatch with Day 10!"
assert len(test_df) == d10_metrics["dataset"]["test_records"], "Test count mismatch with Day 10!"
print("Test set strictly verified against Day 10 split metadata.")

Total Eligible Sales: 131,283 records
Training Set:         105,026 records (2026-01-01 16:21:33 to 2026-08-18 14:17:11)
Untouched Test Set:   26,257 records (2026-08-18 14:17:25 to 2026-10-29 13:59:21)
Test set strictly verified against Day 10 split metadata.


## 3. Leakage-Free Preprocessing
Fit the preprocessing pipeline strictly on `X_train` and transform `X_train` and `X_test` into sparse feature matrices.

In [3]:
X_train, y_train = prepare_features(train_df, return_target=True)
X_test, y_test = prepare_features(test_df, return_target=True)

preprocessor = build_preprocessor(sparse_output=True)
fit_preprocessor(preprocessor, X_train)

X_train_trans = transform_features(preprocessor, X_train)
X_test_trans = transform_features(preprocessor, X_test)

print(f"Transformed X_train shape: {X_train_trans.shape} (Sparse CSR)")
print(f"Transformed X_test shape:  {X_test_trans.shape} (Sparse CSR)")

Transformed X_train shape: (105026, 375) (Sparse CSR)
Transformed X_test shape:  (26257, 375) (Sparse CSR)


## 4. Candidate Model Selection & Architecture
We select **RandomForestRegressor** as the candidate model:
1. **Sparse CSR Compatibility:** Native support for high-dimensional sparse one-hot encoded matrices without dense RAM allocation.
2. **Non-Linear Interaction Modeling:** Tree ensembles capture complex land-to-building ratios, community-tier pricing tiers, and room-count interactions that linear models miss.
3. **Robustness to Extreme Outliers:** Median-like partitioning at tree leaves limits the influence of billion-AED trophy assets on typical unit appraisals.
4. **Sensible Computational Limits:**
   - `n_estimators = 100`: Stable ensemble variance reduction.
   - `max_depth = 12`: Limits leaf memorization and prevents overfitting.
   - `min_samples_leaf = 5`: Regularization against rare categories.
   - `max_features = 'sqrt'`: Fast random sub-sampling across 375 feature columns.
   - `n_jobs = -1`: Fully parallel multi-core execution.

In [4]:
rf_candidate = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

print("Training RandomForestRegressor on 105,026 transactions...")
t_fit_start = time.time()
rf_candidate.fit(X_train_trans, y_train)
fit_duration = time.time() - t_fit_start
print(f"Model training completed in {fit_duration:.2f} seconds.")

Training RandomForestRegressor on 105,026 transactions...


Model training completed in 3.87 seconds.


## 5. Model Evaluation on Untouched Test Set
Generate out-of-sample predictions on the untouched test partition (`26,257` records) and calculate evaluation metrics.

In [5]:
t_pred_start = time.time()
y_pred_rf = rf_candidate.predict(X_test_trans)
pred_duration = time.time() - t_pred_start

# Assert prediction integrity
assert len(y_pred_rf) == len(y_test), "Prediction count mismatch!"
assert np.isfinite(y_pred_rf).all(), "Predictions contain non-finite values!"
assert not np.isnan(y_pred_rf).any(), "Predictions contain NaN values!"

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)

print("=== RANDOM FOREST CANDIDATE MODEL EVALUATION ===")
print(f"Inference Time (26,257 samples): {pred_duration:.2f} seconds ({pred_duration/len(y_test)*1000:.3f} ms/sample)")
print(f"Test MAE:  AED {mae_rf:,.2f}")
print(f"Test RMSE: AED {rmse_rf:,.2f}")

=== RANDOM FOREST CANDIDATE MODEL EVALUATION ===
Inference Time (26,257 samples): 0.11 seconds (0.004 ms/sample)
Test MAE:  AED 1,455,828.87
Test RMSE: AED 8,501,841.09


## 6. Three-Way Benchmark Comparison: Baselines vs Candidate Model
Compare all models on the identical untouched test set.

In [6]:
mae_dummy = d10_metrics["models"]["DummyRegressor_median"]["MAE"]
rmse_dummy = d10_metrics["models"]["DummyRegressor_median"]["RMSE"]
mae_lr = d10_metrics["models"]["LinearRegression_OLS"]["MAE"]
rmse_lr = d10_metrics["models"]["LinearRegression_OLS"]["RMSE"]

mae_gain_vs_dummy = (mae_dummy - mae_rf) / mae_dummy * 100
rmse_gain_vs_dummy = (rmse_dummy - rmse_rf) / rmse_dummy * 100

mae_gain_vs_lr = (mae_lr - mae_rf) / mae_lr * 100
rmse_gain_vs_lr = (rmse_lr - rmse_rf) / rmse_lr * 100

comparison_summary = pd.DataFrame({
    "Model": ["DummyRegressor (Median)", "LinearRegression (OLS)", "RandomForestRegressor (Candidate)"],
    "Test MAE (AED)": [mae_dummy, mae_lr, mae_rf],
    "Test RMSE (AED)": [rmse_dummy, rmse_lr, rmse_rf],
    "MAE Improvement vs Dummy": ["-", f"{(mae_dummy - mae_lr)/mae_dummy*100:+.2f}%", f"{mae_gain_vs_dummy:+.2f}%"],
    "RMSE Improvement vs Dummy": ["-", f"{(rmse_dummy - rmse_lr)/rmse_dummy*100:+.2f}%", f"{rmse_gain_vs_dummy:+.2f}%"]
})

print(comparison_summary.to_string(index=False))

print(f"\nKey Findings:")
print(f"1. RandomForestRegressor reduces MAE by {mae_gain_vs_dummy:.2f}% relative to DummyRegressor (saving AED {mae_dummy - mae_rf:,.2f} per transaction on average).")
print(f"2. Relative to LinearRegression, RandomForestRegressor reduces MAE by {mae_gain_vs_lr:.2f}% (from ~32.3M AED down to ~1.46M AED), completely eliminating the variance catastrophe.")
print(f"3. RMSE is reduced by {rmse_gain_vs_dummy:.2f}% (from 9.82M AED down to 8.50M AED), indicating robust handling of large price variations.")

                            Model  Test MAE (AED)  Test RMSE (AED) MAE Improvement vs Dummy RMSE Improvement vs Dummy
          DummyRegressor (Median)    1.884136e+06     9.815454e+06                        -                         -
           LinearRegression (OLS)    3.229577e+07     1.843406e+08                -1614.09%                 -1778.06%
RandomForestRegressor (Candidate)    1.455829e+06     8.501841e+06                  +22.73%                   +13.38%

Key Findings:
1. RandomForestRegressor reduces MAE by 22.73% relative to DummyRegressor (saving AED 428,307.56 per transaction on average).
2. Relative to LinearRegression, RandomForestRegressor reduces MAE by 95.49% (from ~32.3M AED down to ~1.46M AED), completely eliminating the variance catastrophe.
3. RMSE is reduced by 13.38% (from 9.82M AED down to 8.50M AED), indicating robust handling of large price variations.


## 7. Model Persistence: Creating the End-to-End Pipeline Artifact
Combine the fitted preprocessing transformer and the fitted `RandomForestRegressor` estimator into a single unified scikit-learn `Pipeline` object and persist it via `joblib` for Day 13 inference.

In [7]:
models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)
pipeline_artifact_path = models_dir / "candidate_pipeline.joblib"

# Construct unified Pipeline
full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", rf_candidate)
])

# Save artifact
joblib.dump(full_pipeline, pipeline_artifact_path)
print(f"Saved complete model pipeline artifact to: {pipeline_artifact_path.resolve()}")

# Verify artifact can be loaded and predict directly on raw DataFrame
loaded_pipeline = joblib.load(pipeline_artifact_path)
sample_input = X_test.iloc[:5].copy()
sample_preds = loaded_pipeline.predict(sample_input)

print("\nVerification of loaded pipeline on 5 raw test transactions:")
for i, pred in enumerate(sample_preds):
    actual = y_test.iloc[i]
    print(f"Sample {i+1}: Predicted AED {pred:,.2f} | Actual AED {actual:,.2f} | Error: AED {abs(pred - actual):,.2f}")

Saved complete model pipeline artifact to: C:\Users\nites\Desktop\DubaiEstate-AI-360-Agentic-AI-Powered-Real-Estate-Investment\AI-ML\AI-ML\week02\models\candidate_pipeline.joblib



Verification of loaded pipeline on 5 raw test transactions:
Sample 1: Predicted AED 1,924,439.94 | Actual AED 1,050,000.00 | Error: AED 874,439.94
Sample 2: Predicted AED 2,535,540.50 | Actual AED 2,013,000.00 | Error: AED 522,540.50
Sample 3: Predicted AED 1,649,311.99 | Actual AED 767,000.00 | Error: AED 882,311.99
Sample 4: Predicted AED 2,875,862.31 | Actual AED 173,175.00 | Error: AED 2,702,687.31
Sample 5: Predicted AED 2,080,479.45 | Actual AED 2,500,000.00 | Error: AED 419,520.55


## 8. Exporting Updated Candidate Metrics
Record final candidate model performance and benchmark comparison.

In [8]:
candidate_metrics_payload = {
    "project": "DubaiEstate AI 360 – Agentic AI Powered Real Estate Investment",
    "intern": "Nitesh Kumar Barnwal",
    "domain": "AI/ML",
    "milestone": "Week 2 – Day 11: Candidate Model Training",
    "branch": "feature/week02-aiml-nitesh",
    "target": TARGET_COLUMN,
    "candidate_model": {
        "algorithm": "RandomForestRegressor",
        "parameters": {
            "n_estimators": 100,
            "max_depth": 12,
            "min_samples_leaf": 5,
            "max_features": "sqrt",
            "random_state": 42
        },
        "training_time_seconds": float(fit_duration),
        "test_metrics": {
            "MAE": float(mae_rf),
            "RMSE": float(rmse_rf)
        }
    },
    "comparison_vs_baselines": {
        "MAE_reduction_vs_dummy_pct": float(mae_gain_vs_dummy),
        "RMSE_reduction_vs_dummy_pct": float(rmse_gain_vs_dummy),
        "MAE_reduction_vs_lr_pct": float(mae_gain_vs_lr),
        "RMSE_reduction_vs_lr_pct": float(rmse_gain_vs_lr)
    },
    "verdict": "Candidate RandomForestRegressor successfully improves over all Day 10 baselines."
}

with open("candidate_metrics.json", "w", encoding="utf-8") as f:
    json.dump(candidate_metrics_payload, f, indent=4)

print("Saved candidate_metrics.json successfully.")

Saved candidate_metrics.json successfully.
